# CRISP-DM Walkthrough: From Retail Data to a Forecasting Model

This notebook walks through the six CRISP-DM phases using our retail-demand project. Each phase explains the goal in plain language, then the code shows the work. Run the cells from top to bottom.

**What we are trying to do:** use a product's sales history, calendar, and scheduled price to estimate future daily unit sales for that product in a store.

**CRISP-DM status:** we completed a research version of business understanding, data understanding, data preparation, modeling, and evaluation. Deployment is a working local prototype. A real retailer has not signed off on business targets, hosted the service, or set up ongoing monitoring, so the full real-world CRISP-DM lifecycle is not complete. See the phase summary below.

## CRISP-DM in one picture

Business question → understand the data → prepare examples → train a model → test whether it helps → save and use it. If results are poor, go back and improve an earlier step. CRISP-DM is a cycle, not a one-way checklist.

| Phase | What we did in this project | Status |
| --- | --- | --- |
| Business understanding | Framed the research question as forecasting daily product-store demand; used forecast error as a proxy for usefulness. | Partial: no retailer stakeholder or inventory-cost target was available. |
| Data understanding | Inspected the M5 sales, calendar, and price tables and their date coverage. | Done for this research dataset. |
| Data preparation | Aligned the tables, built history/calendar/price features, and kept the final dates for testing. | Done. |
| Modeling | Selected LightGBM training rounds using training-only validation dates, then refit on the first 70%. | Done. |
| Evaluation | Compared the untouched final 30% with two simple forecasts using WAPE, MAE, RMSE, and bootstrap intervals. | Done. |
| Deployment | Saved a model bundle and demonstrated loading it for a forecast; a local API also exists. | Prototype only: no production hosting, user acceptance, or monitoring. |

The following cells actually run the data-to-model and evaluation steps. The final phase shows how the saved model is applied to a new 28-day request.

In [ ]:
from pathlib import Path
import sys

# Run from the repository root. In Colab, set this to the extracted project directory.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'src' / 'forecast_holdout.py').exists():
    raise FileNotFoundError(f'Project root not found at {PROJECT_DIR}. Set PROJECT_DIR to the repository folder.')
sys.path.insert(0, str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / 'data' / 'raw'
OUTPUT_DIR = PROJECT_DIR / 'artifacts' / 'holdout_70_30'
MAX_SERIES = 0  # 0 = all series; set 1000 for a quick end-to-end run
BOOTSTRAP_REPLICATES = 1000
SEED = 42
print('Project:', PROJECT_DIR)
print('Data:', DATA_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
%pip install -r requirements.txt

In [ ]:
import hashlib
import json
import platform
from datetime import datetime, timezone

import lightgbm as lgb
import numpy as np
import pandas as pd

from src.forecast_experiments import CODES, HORIZON, SEED as MODEL_SEED, TRAIN_DAYS, fit, load_data, make_features, volatility
from src.forecast_holdout import (TRAIN_FRACTION, STATIC, MODELS, compact_feature_dtypes, model_parameters, rolling_holdout_forecasts, summarize_metrics, bootstrap_comparisons)
assert SEED == MODEL_SEED, 'Keep the reproducibility seed aligned with the training helpers.'
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'lightgbm': lgb.__version__})

## Phase 1 — Business understanding

A retailer could use demand forecasts to plan shelf stock or replenishment. We do not have a real retailer's costs, stock records, or business requirements, so this project answers a narrower research question: does the machine-learning forecast reduce unit-sales error compared with simple forecasting rules? Lower error is our measurable proxy for usefulness. It is not proof of lower business cost.

In [ ]:
business_goal = 'Forecast daily unit sales for each product-store series.'
forecast_horizon_days = 28
primary_success_measure = 'Lower holdout WAPE than the four-week weekday-mean baseline.'
print('Goal:', business_goal)
print('Forecast horizon:', forecast_horizon_days, 'days')
print('Research success measure:', primary_success_measure)
print('Business-specific cost target: not provided, so not evaluated.')

## Phase 2 — Data understanding

First we load the three required tables and look at how many product-store histories and dates are available. Each row in the sales file is one product in one store; each `d_...` column is that series' sales on one day. The calendar translates day IDs to dates and provides events and weekdays. The price table gives planned prices by product, store, and week.

In [ ]:
required_files = ['sales_train_validation.csv', 'calendar.csv', 'sell_prices.csv']
missing = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing {missing} under {DATA_DIR}. See README.md for data setup.')
meta, y, codes, days, calendar, sales_file, prices, price_file = load_data(DATA_DIR, MAX_SERIES)
if prices is None:
    raise ValueError('sell_prices.csv is required for this price-aware model.')
print(f'{len(meta):,} series x {len(days):,} daily observations')
print('Sales matrix:', y.shape, '| Price matrix:', prices.shape)
display(meta.head())

In [ ]:
# A quick data-quality and shape check after loading.
print('Rows (product-store series):', len(meta))
print('Daily sales columns:', len(days))
print('Sales values missing:', int(np.isnan(y).sum()))
print('Negative sales values:', int((y < 0).sum()))
print('Share of zero-sales observations:', round(float((y == 0).mean()), 3))
print('Share of missing scheduled prices:', round(float(np.isnan(prices).mean()), 3))
display(meta.head(3))
display(pd.DataFrame({'day_id': days[:5], 'day_of_week': calendar['weekday'].head(5), 'month': calendar['month'].head(5)}))

## Phase 3 — Data preparation

A machine-learning model needs a table of examples: input columns (features) plus the answer for each example (target). We align calendar and prices with each day, encode product/store categories, and later create lag and rolling-sales features. A lag is simply a past value—for example, sales seven days ago.

We split by date instead of randomly. Random splitting would let the model learn from dates that occur after the dates being tested, which makes a time-series score unrealistically optimistic. The earliest 70% of dates are the training period; the latest 30% are kept for the final test.

In [ ]:
split_day = int(np.floor(y.shape[1] * TRAIN_FRACTION))
test_horizon = y.shape[1] - split_day
if split_day < max(TRAIN_DAYS, 52 * 7) + HORIZON or test_horizon < HORIZON:
    raise ValueError('Dataset is too short for the configured training and holdout windows.')
history = y[:, :split_day]
cutoff = split_day - 1
train_days = np.arange(28, split_day, dtype=np.int32)  # first 28 days are feature warm-up
print(f'Train: {days[0]} through {days[cutoff]} ({split_day:,} days, {TRAIN_FRACTION:.0%})')
print(f'Test:  {days[split_day]} through {days[-1]} ({test_horizon:,} days, {1-TRAIN_FRACTION:.0%})')
print(f'Eligible training labels: {len(meta) * len(train_days):,}')

## Phase 4 — Modeling

LightGBM is a collection of decision trees. Each tree makes small corrections to earlier predictions. First we use a short validation slice located entirely within the 70% training period to choose how many trees to build. Then we create a fresh model using all eligible training dates from the 70% partition. The 30% test dates remain unseen by model fitting and round selection.

In [ ]:
selector = fit(history, cutoff, calendar, codes, prices)
selected_rounds = max(1, int(selector.best_iteration or selector.current_iteration()))
del selector
train_features = compact_feature_dtypes(make_features(history, train_days, calendar, codes, prices))
train_labels = history[:, train_days].T.reshape(-1)
train_data = lgb.Dataset(train_features, label=train_labels, categorical_feature=CODES, free_raw_data=True)
model = lgb.train(model_parameters(), train_data, num_boost_round=selected_rounds)
del train_features, train_labels, train_data
print(f'Final model fitted with {selected_rounds} boosting rounds.')

## Phase 5 — Evaluation: make predictions on the held-out dates

A fair evaluation predicts dates the model did not train on. The test period is forecast in non-overlapping windows of up to 28 days. Within one window the model predicts a day, then uses its own prediction as history for the next day. When a later window begins, earlier test-window sales are treated as newly observed history. Actual sales from the current window are never used to predict that same window.

In [ ]:
predictions = rolling_holdout_forecasts(model, y, split_day, test_horizon, calendar, codes, prices)
actual = y[:, split_day:]
test_days = days[split_day:]
volatility_groups, zero_week_share = volatility(history)
print('Prediction arrays:', {name: values.shape for name, values in predictions.items()})

## Phase 5 — Evaluation: compare with simple alternatives

We compare the model with two baselines: repeat sales from the same weekday last week, and average sales for that weekday over the previous four weeks. WAPE tells us aggregate error relative to total demand; MAE is average missed units; RMSE gives extra weight to large misses. Lower is better for all three. Volatility groups tell us whether results differ for steadier and more variable products. Bootstrap intervals show how uncertain the average improvement is across product-store series.

In [ ]:
metrics = summarize_metrics(actual, predictions, volatility_groups, test_days)
series_metrics = pd.DataFrame({
    'series_id': meta['id'].to_numpy(),
    'volatility_group': volatility_groups,
    'zero_week_share': zero_week_share,
    'actual_demand_sum': actual.sum(axis=1),
    **{f'absolute_error_{name}': np.abs(actual - forecast).sum(axis=1) for name, forecast in predictions.items()},
})
bootstrap = bootstrap_comparisons(series_metrics, BOOTSTRAP_REPLICATES)
display(metrics[metrics.volatility_group == 'all'][['model', 'WAPE', 'MAE', 'RMSE']].sort_values('WAPE'))
display(metrics[metrics.volatility_group != 'all'][['model', 'volatility_group', 'WAPE', 'MAE', 'RMSE']])
display(bootstrap)

## Phase 6 — Deployment: save the result

Training does not produce a spreadsheet with a ready-made answer for every possible future day. It produces a saved model file (the learned tree rules) plus metadata that explains the model's features and category codes. The CSV reports record how well it performed on the held-out data. Saving these files is the handoff from training to an application.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
series_metrics.to_csv(OUTPUT_DIR / 'series_metrics.csv', index=False)
bootstrap.to_csv(OUTPUT_DIR / 'bootstrap_comparison.csv', index=False)
model_path = OUTPUT_DIR / 'lightgbm_price.txt'
model.save_model(str(model_path))
category_maps = {code: [str(value) for value in pd.Categorical(meta[source]).categories] for code, source in zip(CODES, STATIC)}
metadata = {
    'model': 'lightgbm_recursive_price', 'model_file': model_path.name,
    'feature_names': model.feature_name(), 'categorical_features': CODES,
    'category_code_maps': category_maps, 'horizon_days': HORIZON, 'history_days_required': 28,
    'selected_boosting_rounds': selected_rounds, 'training_fraction': TRAIN_FRACTION,
    'training_days': split_day, 'test_days': test_horizon, 'series_evaluated': len(meta),
    'seed': MODEL_SEED, 'sales_file_sha256': hashlib.sha256(sales_file.read_bytes()).hexdigest(),
    'sell_prices_file_sha256': hashlib.sha256(price_file.read_bytes()).hexdigest(),
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'lightgbm': lgb.__version__,
}
manifest = {
    'evaluation': 'chronological 70/30 holdout', 'train_fraction': TRAIN_FRACTION,
    'test_fraction': round(1 - TRAIN_FRACTION, 2), 'forecast_window_days': HORIZON,
    'rolling_forecast_origins': int(np.ceil(test_horizon / HORIZON)),
    'train_start_day': days[0], 'train_end_day': days[cutoff],
    'test_start_day': days[split_day], 'test_end_day': days[-1],
    'training_days': split_day, 'test_days': test_horizon,
    'training_observations': int(len(meta) * len(train_days)), 'series_evaluated': len(meta),
    'selected_boosting_rounds': selected_rounds, 'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'seed': MODEL_SEED, 'sales_file_sha256': metadata['sales_file_sha256'],
    'sell_prices_file_sha256': metadata['sell_prices_file_sha256'],
    'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
    'lightgbm': lgb.__version__, 'run_timestamp_utc': datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_DIR / 'model_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved model:', model_path)
print('Saved metrics:', OUTPUT_DIR / 'metrics.csv')

## Phase 6 — Deployment: use the saved model for one new forecast

A forecasting application provides a product/store identity, its latest 28 sales and prices, and the next 28 days of calendar and scheduled-price information. Here we use one real series and the first 28 held-out days to demonstrate the same prediction helper that the API uses. The true test sales are shown only after the forecast, for inspection; they are not given to the model.

A real business rollout still needs data connections, access controls, monitoring, business approval, and a retraining plan. Those operational tasks are not completed by this research prototype.

In [ ]:
from src.forecast_inference import forecast_series, load_model_bundle
saved_model, saved_metadata = load_model_bundle(OUTPUT_DIR)
example_series = 0
example_forecast = forecast_series(
    saved_model, saved_metadata,
    sales_history=y[example_series, split_day - 28:split_day],
    static_metadata=meta.iloc[example_series][STATIC].to_dict(),
    future_calendar=calendar.iloc[split_day:split_day + 28].to_dict(orient='records'),
    recent_prices=prices[example_series, split_day - 28:split_day],
    future_prices=prices[example_series, split_day:split_day + 28],
)
example_result = pd.DataFrame({
    'day_id': days[split_day:split_day + 28],
    'actual_units': actual[example_series, :28],
    'predicted_units': example_forecast,
})
print('Example product/store:', meta.iloc[example_series]['id'])
display(example_result.head(10))

## What the result of the whole process looks like

The main output is a folder named `artifacts/holdout_70_30/`. It contains the saved model that can be loaded later, metadata describing its inputs, score tables, bootstrap comparisons, and a run manifest recording the split and data checksums. The score describes performance on historical unseen dates; it is not a guarantee for future dates or another retailer.

A forecast is a decision aid. An inventory system or staff member could consider the predicted units when planning stock. This prototype does not place orders or change inventory automatically.

## Run instructions

From the repository root, open this notebook in Jupyter and run all cells. On Windows, create `.venv_cp312` with `py -3.12 -m venv .venv_cp312`, activate it, then install `requirements.txt`. In Colab, upload or extract the project and `data/raw` files, set `PROJECT_DIR`, then run all cells. Full-population training needs substantial RAM; use `MAX_SERIES = 1000` to check the complete pipeline first.